# NetShield — Phase 5: Modeling Experiments

## Cross-Dataset Generalization and Robustness

The central research question is:

> **Can a model trained on one network dataset generalize to another dataset, and can a semantically aligned representation improve that cross-dataset robustness?**

This notebook establishes controlled baselines first, then tests targeted robustness interventions.

### Rules
- No identifiers, IPs, timestamps, or `source_file` as predictive features.
- Preprocessing is fitted on training data only.
- Destination cross-dataset data are treated as untouched evaluation environments.
- Accuracy is not the primary metric.
- No default SMOTE.
- No blind outlier deletion or zero replacement.
- No tuning based on the final destination test result.


## Experiment map

```text
A. UNSW native:       UNSW train → UNSW test
B. CICIDS native:     CICIDS train → CICIDS test

C. UNSW → CICIDS:     common semantic features
D. CICIDS → UNSW:     common semantic features

E. Measure within-vs-cross gaps

F. Targeted interventions
   - class weighting
   - scaling sensitivity
   - carefully justified transformations
   - feature/redundancy sensitivity

G. Error analysis
   - confusion matrices
   - PR-AUC / F1 / Macro-F1 / FPR
   - feature importance
   - attack-family analysis where justified
```

The first run deliberately establishes baselines before optimization.


In [1]:
# 0. Imports and project paths

from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_score, recall_score, f1_score, average_precision_score,
    confusion_matrix, roc_auc_score
)

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_ROOT = PROJECT_ROOT / "data" / "raw"
UNSW_ROOT = RAW_ROOT / "UNSW-NB15"
CICIDS_ROOT = RAW_ROOT / "CICIDS2017"

OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "modeling_experiments"
TABLE_DIR = OUTPUT_ROOT / "tables"
PLOT_DIR = OUTPUT_ROOT / "plots"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("UNSW:", UNSW_ROOT)
print("CICIDS:", CICIDS_ROOT)


Project root: c:\Users\BISMILLAH\Documents\NetShield
UNSW: c:\Users\BISMILLAH\Documents\NetShield\data\raw\UNSW-NB15
CICIDS: c:\Users\BISMILLAH\Documents\NetShield\data\raw\CICIDS2017


In [2]:
# 1. Load official UNSW-NB15 train/test

unsw_train_path = UNSW_ROOT / "Training and Testing Sets" / "UNSW_NB15_training-set.csv"
unsw_test_path = UNSW_ROOT / "Training and Testing Sets" / "UNSW_NB15_testing-set.csv"

if not unsw_train_path.exists() or not unsw_test_path.exists():
    raise FileNotFoundError(
        "Expected UNSW official training/testing files were not found. "
        "Check data/raw/UNSW-NB15/Training and Testing Sets/"
    )

unsw_train = pd.read_csv(unsw_train_path)
unsw_test = pd.read_csv(unsw_test_path)

print("UNSW train:", unsw_train.shape)
print("UNSW test :", unsw_test.shape)


UNSW train: (175341, 45)
UNSW test : (82332, 45)


In [3]:
# 2. Load CICIDS2017 CSV files

cicids_files = sorted(CICIDS_ROOT.rglob("*.csv"))
if not cicids_files:
    raise FileNotFoundError(f"No CICIDS CSV files found under {CICIDS_ROOT}")

parts = []
for path in cicids_files:
    part = pd.read_csv(path, low_memory=False)
    part.columns = part.columns.str.strip()
    part["source_file"] = path.name
    parts.append(part)

cicids = pd.concat(parts, ignore_index=True)

print("CICIDS:", cicids.shape)
print("CSV files:", len(cicids_files))


CICIDS: (2830743, 80)
CSV files: 8


## 2. Targets and native feature spaces

Primary task:

- `0` = benign
- `1` = attack

The CICIDS `source_file` is retained only as metadata. It is not a model feature.


In [5]:
# 3. Binary labels
# IMPORTANT: Do not copy the huge CICIDS dataframe.
# We add only the new label column directly.

# UNSW labels
unsw_train["label_binary"] = pd.to_numeric(
    unsw_train["label"], errors="coerce"
).astype("int8")

unsw_test["label_binary"] = pd.to_numeric(
    unsw_test["label"], errors="coerce"
).astype("int8")

# CICIDS labels
cicids["label_binary"] = (
    cicids["Label"]
    .astype(str)
    .str.strip()
    .str.upper()
    .ne("BENIGN")
    .astype("int8")
)

print("UNSW train proportions:")
print(
    unsw_train["label_binary"]
    .value_counts(normalize=True)
    .sort_index()
)

print("\nUNSW test proportions:")
print(
    unsw_test["label_binary"]
    .value_counts(normalize=True)
    .sort_index()
)

print("\nCICIDS proportions:")
print(
    cicids["label_binary"]
    .value_counts(normalize=True)
    .sort_index()
)

UNSW train proportions:
label_binary
0    0.319378
1    0.680622
Name: proportion, dtype: float64

UNSW test proportions:
label_binary
0    0.4494
1    0.5506
Name: proportion, dtype: float64

CICIDS proportions:
label_binary
0    0.803004
1    0.196996
Name: proportion, dtype: float64


In [6]:
# 4. Native predictive features

UNSW_DROP = {"id", "label", "label_binary", "attack_cat"}
CICIDS_DROP = {
    "Label", "label_binary", "source_file",
    "Flow ID", "Source IP", "Destination IP", "Timestamp"
}

unsw_native_features = [
    c for c in unsw_train.columns
    if c not in UNSW_DROP and pd.api.types.is_numeric_dtype(unsw_train[c])
]

cicids_native_features = [
    c for c in cicids.columns
    if c not in CICIDS_DROP and pd.api.types.is_numeric_dtype(cicids[c])
]

print("UNSW native numeric features:", len(unsw_native_features))
print("CICIDS native numeric features:", len(cicids_native_features))


UNSW native numeric features: 39
CICIDS native numeric features: 78


## 3. Leakage-safe baseline models

The imputer/scaler lives inside the sklearn pipeline, so it is fitted only on the data passed to `.fit()`.

Models:
- Logistic Regression
- Decision Tree
- Random Forest
- Gradient Boosting (`HistGradientBoostingClassifier`)


In [7]:
# 5. Baseline model pipelines

models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
    ]),
    "Decision Tree": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", DecisionTreeClassifier(random_state=RANDOM_STATE))
    ]),
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1
        ))
    ]),
    "Gradient Boosting": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", HistGradientBoostingClassifier(
            max_iter=200, random_state=RANDOM_STATE
        ))
    ])
}

print(list(models.keys()))


['Logistic Regression', 'Decision Tree', 'Random Forest', 'Gradient Boosting']


In [8]:
# 6. Evaluation helper

def evaluate_binary_model(model, X_train, y_train, X_test, y_test,
                          experiment, model_name):
    fitted = clone(model)
    fitted.fit(X_train, y_train)

    y_pred = fitted.predict(X_test)
    y_score = (
        fitted.predict_proba(X_test)[:, 1]
        if hasattr(fitted, "predict_proba") else None
    )

    tn, fp, fn, tp = confusion_matrix(
        y_test, y_pred, labels=[0, 1]
    ).ravel()

    return {
        "experiment": experiment,
        "model": model_name,
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "macro_f1": f1_score(y_test, y_pred, average="macro", zero_division=0),
        "pr_auc": (
            average_precision_score(y_test, y_score)
            if y_score is not None else np.nan
        ),
        "roc_auc": (
            roc_auc_score(y_test, y_score)
            if y_score is not None and len(np.unique(y_test)) == 2 else np.nan
        ),
        "fpr": fp / (fp + tn) if (fp + tn) else np.nan,
        "tn": tn, "fp": fp, "fn": fn, "tp": tp
    }, fitted


# 4. Experiment A — UNSW native within-dataset baseline

Train on the official UNSW training set and evaluate on the official UNSW testing set.


In [9]:
# 7. UNSW native baseline

X_unsw_train = unsw_train[unsw_native_features].replace([np.inf, -np.inf], np.nan)
X_unsw_test = unsw_test[unsw_native_features].replace([np.inf, -np.inf], np.nan)
y_unsw_train = unsw_train["label_binary"]
y_unsw_test = unsw_test["label_binary"]

unsw_rows = []
unsw_fitted = {}

for name, model in models.items():
    row, fitted = evaluate_binary_model(
        model, X_unsw_train, y_unsw_train,
        X_unsw_test, y_unsw_test,
        "UNSW_native_within", name
    )
    unsw_rows.append(row)
    unsw_fitted[name] = fitted

unsw_results = pd.DataFrame(unsw_rows)
display(unsw_results.sort_values("f1", ascending=False))
unsw_results.to_csv(TABLE_DIR / "01_unsw_native_within_results.csv", index=False)


,experiment,model,precision,recall,f1,macro_f1,pr_auc,roc_auc,fpr,tn,fp,fn,tp
2,UNSW_native_within,Random Forest,0.817485,0.986169,0.893939,0.864923,0.981270,0.977786,0.269757,27019,9981,627,44705
3,UNSW_native_within,Gradient Boosting,0.819317,0.982794,0.893640,0.865187,0.987489,0.982988,0.265541,27175,9825,780,44552
1,UNSW_native_within,Decision Tree,0.821952,0.954822,0.883419,0.856036,0.812454,0.853058,0.253405,27624,9376,2048,43284
0,UNSW_native_within,Logistic Regression,0.729555,0.942844,0.822599,0.759564,0.904326,0.888205,0.428216,21156,15844,2591,42741


# 5. Experiment B — CICIDS native within-dataset baseline

This first CICIDS baseline uses a reproducible stratified 80/20 holdout.

Later we will add a source-file-aware robustness evaluation because CICIDS capture files correspond to different traffic/attack scenarios.


In [10]:
# 8. CICIDS baseline split

cicids_model = cicids.dropna(subset=["label_binary"]).copy()

X_cicids = cicids_model[cicids_native_features].replace([np.inf, -np.inf], np.nan)
y_cicids = cicids_model["label_binary"]

X_cicids_train, X_cicids_test, y_cicids_train, y_cicids_test = train_test_split(
    X_cicids, y_cicids,
    test_size=0.20,
    stratify=y_cicids,
    random_state=RANDOM_STATE
)

print("CICIDS train:", X_cicids_train.shape)
print("CICIDS test :", X_cicids_test.shape)


MemoryError: Unable to allocate 21.6 MiB for an array with shape (1, 2830743) and data type int64

In [ ]:
# 9. CICIDS native baseline

cicids_rows = []
cicids_fitted = {}

for name, model in models.items():
    row, fitted = evaluate_binary_model(
        model, X_cicids_train, y_cicids_train,
        X_cicids_test, y_cicids_test,
        "CICIDS_native_within", name
    )
    cicids_rows.append(row)
    cicids_fitted[name] = fitted

cicids_results = pd.DataFrame(cicids_rows)
display(cicids_results.sort_values("f1", ascending=False))
cicids_results.to_csv(TABLE_DIR / "02_cicids_native_within_results.csv", index=False)


# 6. Common semantic representation

Direct raw-feature transfer is not valid because UNSW-NB15 and CICIDS2017 have different predictive schemas.

We therefore use the semantically aligned representation developed in Phase 4:

```text
flow_duration_sec
fwd_packets
bwd_packets
fwd_bytes
bwd_bytes
total_packets
total_bytes
packet_rate
fwd_packet_rate
bwd_packet_rate
avg_packet_size
```

The mapping is based on feature meaning and units, not merely similar names.


In [ ]:
# 10. Common semantic feature builders

COMMON_FEATURES = [
    "flow_duration_sec", "fwd_packets", "bwd_packets",
    "fwd_bytes", "bwd_bytes", "total_packets", "total_bytes",
    "packet_rate", "fwd_packet_rate", "bwd_packet_rate",
    "avg_packet_size"
]

def safe_divide_series(numerator, denominator):
    numerator = pd.to_numeric(numerator, errors="coerce")
    if np.isscalar(denominator):
        denominator = pd.Series(denominator, index=numerator.index)
    else:
        denominator = pd.to_numeric(denominator, errors="coerce")
    denominator = denominator.replace(0, np.nan)
    return numerator.div(denominator)

def build_unsw_common(df):
    out = pd.DataFrame(index=df.index)
    out["flow_duration_sec"] = pd.to_numeric(df["dur"], errors="coerce")
    out["fwd_packets"] = pd.to_numeric(df["spkts"], errors="coerce")
    out["bwd_packets"] = pd.to_numeric(df["dpkts"], errors="coerce")
    out["fwd_bytes"] = pd.to_numeric(df["sbytes"], errors="coerce")
    out["bwd_bytes"] = pd.to_numeric(df["dbytes"], errors="coerce")
    out["total_packets"] = out["fwd_packets"] + out["bwd_packets"]
    out["total_bytes"] = out["fwd_bytes"] + out["bwd_bytes"]
    out["packet_rate"] = safe_divide_series(out["total_packets"], out["flow_duration_sec"])
    out["fwd_packet_rate"] = safe_divide_series(out["fwd_packets"], out["flow_duration_sec"])
    out["bwd_packet_rate"] = safe_divide_series(out["bwd_packets"], out["flow_duration_sec"])
    out["avg_packet_size"] = safe_divide_series(out["total_bytes"], out["total_packets"])
    out["label_binary"] = df["label_binary"].astype("int8")
    return out

def build_cicids_common(df):
    out = pd.DataFrame(index=df.index)
    # CICIDS Flow Duration is in microseconds.
    out["flow_duration_sec"] = safe_divide_series(df["Flow Duration"], 1_000_000)
    out["fwd_packets"] = pd.to_numeric(df["Total Fwd Packets"], errors="coerce")
    out["bwd_packets"] = pd.to_numeric(df["Total Backward Packets"], errors="coerce")
    out["fwd_bytes"] = pd.to_numeric(df["Total Length of Fwd Packets"], errors="coerce")
    out["bwd_bytes"] = pd.to_numeric(df["Total Length of Bwd Packets"], errors="coerce")
    out["total_packets"] = out["fwd_packets"] + out["bwd_packets"]
    out["total_bytes"] = out["fwd_bytes"] + out["bwd_bytes"]
    out["packet_rate"] = pd.to_numeric(df["Flow Packets/s"], errors="coerce")
    out["fwd_packet_rate"] = pd.to_numeric(df["Fwd Packets/s"], errors="coerce")
    out["bwd_packet_rate"] = pd.to_numeric(df["Bwd Packets/s"], errors="coerce")
    out["avg_packet_size"] = pd.to_numeric(df["Average Packet Size"], errors="coerce")
    out["label_binary"] = df["label_binary"].astype("int8")
    return out

unsw_common_train = build_unsw_common(unsw_train)
unsw_common_test = build_unsw_common(unsw_test)
cicids_common = build_cicids_common(cicids_model)

print("UNSW common train:", unsw_common_train.shape)
print("UNSW common test :", unsw_common_test.shape)
print("CICIDS common    :", cicids_common.shape)


# 7. Experiment C — UNSW → CICIDS

Train on UNSW common features. Test directly on CICIDS common features.

There is **no fitting on CICIDS**.


In [ ]:
# 11. UNSW -> CICIDS common-feature transfer

X_unsw_common_train = unsw_common_train[COMMON_FEATURES].replace([np.inf, -np.inf], np.nan)
y_unsw_common_train = unsw_common_train["label_binary"]

X_cicids_common = cicids_common[COMMON_FEATURES].replace([np.inf, -np.inf], np.nan)
y_cicids_common = cicids_common["label_binary"]

unsw_to_cicids_rows = []
unsw_to_cicids_fitted = {}

for name, model in models.items():
    row, fitted = evaluate_binary_model(
        model, X_unsw_common_train, y_unsw_common_train,
        X_cicids_common, y_cicids_common,
        "UNSW_to_CICIDS_common", name
    )
    unsw_to_cicids_rows.append(row)
    unsw_to_cicids_fitted[name] = fitted

unsw_to_cicids = pd.DataFrame(unsw_to_cicids_rows)
display(unsw_to_cicids.sort_values("f1", ascending=False))
unsw_to_cicids.to_csv(TABLE_DIR / "03_unsw_to_cicids_common_results.csv", index=False)


# 8. Experiment D — CICIDS → UNSW

Train on CICIDS common features. Test on the official UNSW test environment.

Again, the destination dataset is not used for fitting preprocessing or model parameters.


In [ ]:
# 12. CICIDS -> UNSW common-feature transfer

X_cicids_common = cicids_common[COMMON_FEATURES].replace([np.inf, -np.inf], np.nan)
y_cicids_common = cicids_common["label_binary"]

X_unsw_common_test = unsw_common_test[COMMON_FEATURES].replace([np.inf, -np.inf], np.nan)
y_unsw_common_test = unsw_common_test["label_binary"]

cicids_to_unsw_rows = []
cicids_to_unsw_fitted = {}

for name, model in models.items():
    row, fitted = evaluate_binary_model(
        model, X_cicids_common, y_cicids_common,
        X_unsw_common_test, y_unsw_common_test,
        "CICIDS_to_UNSW_common", name
    )
    cicids_to_unsw_rows.append(row)
    cicids_to_unsw_fitted[name] = fitted

cicids_to_unsw = pd.DataFrame(cicids_to_unsw_rows)
display(cicids_to_unsw.sort_values("f1", ascending=False))
cicids_to_unsw.to_csv(TABLE_DIR / "04_cicids_to_unsw_common_results.csv", index=False)


# 9. Compare within-vs-cross performance

The key quantity is the generalization gap:

```text
within-source F1 − cross-dataset F1
```

A large gap is not a failure. It is evidence of dataset-dependent behavior that we will try to explain and reduce.


In [ ]:
# 13. Combined baseline table

all_results = pd.concat([
    unsw_results, cicids_results,
    unsw_to_cicids, cicids_to_unsw
], ignore_index=True)

display(all_results[
    ["experiment", "model", "precision", "recall", "f1",
     "macro_f1", "pr_auc", "fpr"]
].sort_values(["model", "experiment"]))

all_results.to_csv(TABLE_DIR / "05_all_baseline_results.csv", index=False)


In [ ]:
# 14. Generalization gaps

gap_rows = []

for model_name in models:
    unsw_within = unsw_results.set_index("model").loc[model_name, "f1"]
    cicids_within = cicids_results.set_index("model").loc[model_name, "f1"]
    u2c = unsw_to_cicids.set_index("model").loc[model_name, "f1"]
    c2u = cicids_to_unsw.set_index("model").loc[model_name, "f1"]

    gap_rows.append({
        "model": model_name,
        "direction": "UNSW_to_CICIDS",
        "within_source_f1": unsw_within,
        "cross_dataset_f1": u2c,
        "f1_gap": unsw_within - u2c
    })
    gap_rows.append({
        "model": model_name,
        "direction": "CICIDS_to_UNSW",
        "within_source_f1": cicids_within,
        "cross_dataset_f1": c2u,
        "f1_gap": cicids_within - c2u
    })

gaps = pd.DataFrame(gap_rows)
display(gaps)
gaps.to_csv(TABLE_DIR / "06_cross_dataset_generalization_gaps.csv", index=False)


# 10. Targeted intervention — class weighting

The datasets have very different binary class priors. We therefore test `class_weight="balanced"` as a targeted, interpretable intervention.

This is **not** assumed to be better. We measure whether it improves transfer.

Gradient Boosting is not included in this specific intervention because the chosen sklearn implementation does not expose the same simple `class_weight` parameter.


In [ ]:
# 15. Class-weighted transfer models

weighted_models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE
        ))
    ]),
    "Decision Tree": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", DecisionTreeClassifier(
            class_weight="balanced", random_state=RANDOM_STATE
        ))
    ]),
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=200, class_weight="balanced",
            random_state=RANDOM_STATE, n_jobs=-1
        ))
    ])
}

weighted_rows = []

for name, model in weighted_models.items():
    row, _ = evaluate_binary_model(
        model, X_unsw_common_train, y_unsw_common_train,
        X_cicids_common, y_cicids_common,
        "UNSW_to_CICIDS_common_class_weighted", name
    )
    weighted_rows.append(row)

for name, model in weighted_models.items():
    row, _ = evaluate_binary_model(
        model, X_cicids_common, y_cicids_common,
        X_unsw_common_test, y_unsw_common_test,
        "CICIDS_to_UNSW_common_class_weighted", name
    )
    weighted_rows.append(row)

weighted_results = pd.DataFrame(weighted_rows)
display(weighted_results[
    ["experiment", "model", "precision", "recall", "f1",
     "macro_f1", "pr_auc", "fpr"]
].sort_values(["experiment", "f1"], ascending=[True, False]))

weighted_results.to_csv(
    TABLE_DIR / "07_class_weighted_transfer_results.csv", index=False
)


# 11. Important: do not tune on the destination test set

Stop after the baseline + class-weighted experiments and inspect the evidence.

The final destination dataset in a cross-dataset experiment is an external evaluation environment. We must not choose transformations, hyperparameters, thresholds, or features because they improve that final destination result.

Future interventions will be motivated by the observed problems and evaluated with an appropriate train/validation design.


In [ ]:
# 16. Save an auditable experiment manifest

manifest = {
    "project": "NetShield",
    "phase": "05_modeling_experiments",
    "random_state": RANDOM_STATE,
    "primary_task": "binary intrusion detection",
    "primary_metrics": [
        "precision", "recall", "f1", "macro_f1", "pr_auc", "fpr"
    ],
    "within_dataset": [
        "UNSW_native_within",
        "CICIDS_native_within"
    ],
    "cross_dataset": [
        "UNSW_to_CICIDS_common",
        "CICIDS_to_UNSW_common"
    ],
    "common_features": COMMON_FEATURES,
    "first_intervention": "class_weight_balanced",
    "destination_test_rule": "not used for tuning",
    "next_stage": [
        "distribution-shift vs performance-drop analysis",
        "carefully justified skew transformations",
        "scaling sensitivity",
        "feature redundancy sensitivity",
        "source-file-aware CICIDS evaluation",
        "confusion matrices and error analysis",
        "feature importance"
    ]
}

with open(OUTPUT_ROOT / "experiment_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("Saved experiment manifest.")


# 12. What comes next?

We will **not** add random techniques just to make the F1 score larger.

The next stage will use evidence from these results to decide which interventions are justified:

1. Distribution-aware transformations for strongly skewed common features.
2. Scaling sensitivity.
3. Feature redundancy sensitivity.
4. Source-file-aware CICIDS evaluation to test whether random splitting is optimistic.
5. Confusion matrices and error analysis.
6. Feature importance.
7. Attack-family transfer analysis where the label taxonomy supports a defensible comparison.

The final research result should answer:

> **Which design choices genuinely improve cross-dataset robustness, by how much, and why?**

A lower score can still be a strong scientific result if it reveals a real limitation and we can explain it.
